# LoRA-Edit: First-Frame-Guided Video Editing

This notebook provides an end-to-end workflow for using [LoRA-Edit](https://github.com/cjeen/LoRAEdit) to edit a video based on a modified first frame.

**Workflow:**
1.  **Setup:** Clone the repository and install all dependencies.
2.  **Download Models:** Download the necessary pre-trained models.
3.  **Data Preparation:** Use the interactive UI to preprocess your video.
4.  **Training:** Fine-tune a LoRA adapter on your specific video and object.
5.  **Inference:** Provide an edited version of the first frame to guide the video generation.

Let's get started!

## 1.1. Clone Repository

In [ ]:
!git clone --recurse-submodules https://github.com/cjeen/LoRAEdit.git
%cd LoRAEdit

## 1.2. Install Dependencies

Next, we install all the required Python packages. This step will take a few minutes.

In [ ]:
!pip install -r requirements.txt
!pip install xformers

# 2. Download Pre-trained Models

This project relies on three main pre-trained models:
1.  **Wan2.1-I2V**: The core Image-to-Video model that we will fine-tune.
2.  **SAM2 (Segment Anything Model 2)**: Used for object tracking to create masks.
3.  **Florence-2**: A vision-language model used to automatically generate captions for images.

The following cells will download these models. This will take some time and consume a significant amount of disk space.

In [ ]:
# Download the Wan2.1-I2V model from Hugging Face
# This is a large model (approx. 28 GB)
!pip install huggingface_hub
!huggingface-cli download Wan-AI/Wan2.1-I2V-14B-480P --local-dir ./Wan2.1-I2V-14B-480P --local-dir-use-symlinks False

# Verify download
import os
if os.path.exists('./Wan2.1-I2V-14B-480P/Wan2.1_VAE.pth'):
    print('✅ Wan2.1-I2V model downloaded successfully.')
else:
    print('❌ Wan2.1-I2V model download failed.')

In [ ]:
# Download the SAM2 model checkpoint
!mkdir -p models_sam
!wget https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt -O models_sam/sam2_hiera_large.pt

# Verify download
import os
if os.path.exists('models_sam/sam2_hiera_large.pt'):
    print('✅ SAM2 model downloaded successfully.')
else:
    print('❌ SAM2 model download failed.')

In [ ]:
# Download and cache the Florence-2 model
# This ensures the model is available for later steps
from transformers import AutoProcessor, AutoModelForCausalLM

print('Downloading Florence-2 model...')
try:
    model_id = 'multimodalart/Florence-2-large-no-flash-attn'
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
    print('✅ Florence-2 model downloaded successfully.')
except Exception as e:
    print(f'❌ Florence-2 model download failed: {e}')

# 3. Data Preparation via Interactive UI

This section launches an interactive web UI that you will use to prepare your data. This is the same interface you would use if running the project locally.

**Instructions:**
1.  Run the cell below. It will start the Gradio server and print a public URL (it will look like `https://....gradio.live`).
2.  **Click the public URL** to open the UI in a new tab.
3.  Inside the UI:
    a. Go to the **Upload Video** tab and upload your video. Set the desired frame count and resolution, then click **Extract Frames**.
    b. Once frames are loaded, you will see the first frame. Click on the object you want to track to add positive points.
    c. Click **Submit Mask for Tracking**. This will track the object through the video.
    d. **IMPORTANT**: In the **Data Processing Save Path** text box, make sure the path is set to `processed_data/my_awesome_video`. You can change `my_awesome_video` to whatever you like, but make sure it's consistent in the later steps of this notebook.
    e. In the **Model Checkpoint Path**, make sure it is set to the correct path: `/content/LoRAEdit/Wan2.1-I2V-14B-480P`.
    f. Click **Process and Save Data**.
4.  Once you have finished and the data is saved, you can **stop the cell below** by clicking the stop button, and then proceed to Section 4.

In [ ]:
# Launch the Gradio UI for preprocessing
!python predata_app.py --checkpoint_dir models_sam/sam2_hiera_large.pt

# 4. LoRA Training

Now we are ready to train the LoRA (Low-Rank Adaptation) model. This process fine-tunes the giant Wan2.1-I2V model on our specific video sequence, teaching it about the object we masked.

**⚠️ High VRAM Warning:** This step is computationally intensive and requires a lot of GPU memory. The default settings (49 frames) require ~22GB of VRAM, which is more than a free Google Colab T4 GPU (~15GB) can provide.
*   If you are on a free tier, you will likely get a 'CUDA out of memory' error.
*   To run on a free tier, you should go back to **Step 3.2** and change `TARGET_FRAMES` to a smaller number (e.g., 21 or 13) and re-run the preprocessing steps.
*   This notebook is best run on a Colab Pro subscription with access to A100 or V100 GPUs.

The training command below is taken from the official `README`. It uses `deepspeed` for memory-efficient training. Training will take a while (the paper reports ~30-50 minutes for 100 epochs on an RTX 4090).

In [ ]:
import os

# --- Configuration ---
# IMPORTANT: This must match the sequence name you used in the UI!
sequence_name = 'my_awesome_video'
# --- End Configuration ---

training_config_path = f'processed_data/{sequence_name}/configs/training.toml'

if not os.path.exists(training_config_path):
    print(f"❌ Error: Training config file not found at '{training_config_path}'")
    print('Please make sure you have successfully run all the steps in Section 3.')
else:
    print(f'Found training config: {training_config_path}')
    print('Starting LoRA training... This will take a long time.')
    !NCCL_P2P_DISABLE="1" NCCL_IB_DISABLE="1" deepspeed --num_gpus=1 train.py --deepspeed --config {training_config_path}

    print('\n✅ Training finished!')

# 5. Inference: Generate the Edited Video

The training is complete! Now for the fun part.

Take the original first frame (`processed_data/my_awesome_video/source_frames/00000.png`), and edit it however you like. For example, change the color of an object, add an accessory, etc.

Then, upload your edited first frame below. The model will use this image as a guide to edit the entire video.

In [ ]:
from google.colab import files
import os
from PIL import Image

# --- Configuration ---
sequence_name = 'my_awesome_video'
# --- End Configuration ---

data_dir = f'processed_data/{sequence_name}'

print('Please upload your edited first frame (e.g., with a color change).')
uploaded_edit = files.upload()

if not uploaded_edit:
    print('No file uploaded. Please run the cell again and select your edited frame.')
else:
    edited_filename = list(uploaded_edit.keys())[0]
    output_path = os.path.join(data_dir, 'edited_image.png')
    Image.open(edited_filename).save(output_path)
    if edited_filename != 'edited_image.png':
        os.remove(edited_filename)

    print(f"✅ Edited frame '{edited_filename}' uploaded and saved as '{output_path}'")

In [ ]:
import os

# --- Configuration ---
sequence_name = 'my_awesome_video'
wan_model_path = './Wan2.1-I2V-14B-480P'
# --- End Configuration ---

data_dir = f'processed_data/{sequence_name}'

if not os.path.exists(os.path.join(data_dir, 'edited_image.png')):
    print('❌ Error: Edited image not found.')
    print('Please make sure you have successfully run the previous cell to upload it.')
else:
    print('Starting inference... This may take a few minutes.')
    !python inference.py --model_root_dir {wan_model_path} --data_dir {data_dir}
    print('\n✅ Inference finished!')

## 5.1. View Your Edited Video!

If everything worked, your final edited video should be displayed below. You can also find it in the file browser at `processed_data/my_awesome_video/edited_video.mp4`.

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# --- Configuration ---
sequence_name = 'my_awesome_video'
# --- End Configuration ---

video_path = f'processed_data/{sequence_name}/edited_video.mp4'

if not os.path.exists(video_path):
    print('❌ Error: Final video not found. Something went wrong during inference.')
else:
    mp4 = open(video_path,'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'''
    <video width=400 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    '''))